# Ingeniería del Dato — Heart Disease UCI
## TFG: Sistema Multimodal de IA para Diagnóstico Médico
**Autor:** Alfredo Pérez Navalón | **Universidad:** UFV

> **IMPORTANTE:** Ejecutar todas las celdas en orden secuencial (Run All).

---

### 1. Origen de los datos
Los datos provienen del **UCI Machine Learning Repository**, combinando **4 bases hospitalarias** independientes:

| Hospital | Ubicación | Investigador principal |
|----------|-----------|------------------------|
| Cleveland Clinic | Ohio, USA | Dr. Robert Detrano |
| Hungarian Institute of Cardiology | Budapest, Hungría | Dr. Andras Janosi |
| University Hospital | Zúrich, Suiza | Dr. William Steinbrunn |
| VA Medical Center | Long Beach, California, USA | Dr. Matthew Pfisterer |

**Referencia:** Janosi, A., Steinbrunn, W., Pfisterer, M., & Detrano, R. (1988). Heart Disease [Dataset]. UCI ML Repository.

Se utilizan las versiones procesadas con **14 atributos clínicos** seleccionados por los investigadores originales de un total de 76 variables. Los archivos originales (.data) no contienen cabeceras y utilizan `?` como marcador de valores ausentes.

> **Nota sobre versión Kaggle:** Existe una versión popular en Kaggle (1025 registros) que contiene **723 filas duplicadas (~70%)**, invalidándola para entrenamiento fiable. Se opta por las fuentes originales UCI.

In [ ]:
# ============================================
# IMPORTS Y CONFIGURACIÓN
# ============================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

NAVY = 'midnightblue'
BLUE = 'steelblue'
ORANGE = 'darkorange'
RED = 'crimson'
TEAL = 'teal'
GREEN = 'forestgreen'
LIGHT_BLUE = 'lightsteelblue'
PALETTE = [NAVY, TEAL, ORANGE, RED]

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'

# Crear carpetas de salida
os.makedirs('../reports/figures_heart/', exist_ok=True)
os.makedirs('../data/processed/', exist_ok=True)

FIGURES = '../reports/figures_heart/'
print('Configuración cargada.')

In [ ]:
# ============================================
# CARGA Y COMBINACIÓN DE LAS 4 BASES
# ============================================
columns = ['age','sex','cp','trestbps','chol','fbs','restecg',
           'thalach','exang','oldpeak','slope','ca','thal','num']

df_cleveland = pd.read_csv('../data/raw/heart-disease/processed.cleveland.data',
                           names=columns, na_values='?')
df_hungarian = pd.read_csv('../data/raw/heart-disease/processed.hungarian.data',
                           names=columns, na_values='?')
df_switzerland = pd.read_csv('../data/raw/heart-disease/processed.switzerland.data',
                             names=columns, na_values='?')
df_va = pd.read_csv('../data/raw/heart-disease/processed.va.data',
                     names=columns, na_values='?')

df_cleveland['source'] = 'Cleveland'
df_hungarian['source'] = 'Hungarian'
df_switzerland['source'] = 'Switzerland'
df_va['source'] = 'VA Long Beach'

df_raw = pd.concat([df_cleveland, df_hungarian, df_switzerland, df_va],
                   ignore_index=True)

print('=== REGISTROS POR HOSPITAL ===')
print(df_raw['source'].value_counts())
print(f'\nTotal combinado: {df_raw.shape[0]} registros, {df_raw.shape[1]} columnas')

---
### 2. Exploración inicial

In [ ]:
# ============================================
# DICCIONARIO CLÍNICO
# ============================================
data_dict = pd.DataFrame({
    'Variable': columns,
    'Descripción': [
        'Edad del paciente (años)',
        'Sexo (1=hombre, 0=mujer)',
        'Tipo de dolor torácico (1=angina típica, 2=atípica, 3=no anginoso, 4=asintomático)',
        'Presión arterial en reposo (mmHg)',
        'Colesterol sérico (mg/dl)',
        'Glucemia en ayunas > 120 mg/dl (1=sí, 0=no)',
        'Resultado ECG en reposo (0=normal, 1=anomalía ST-T, 2=hipertrofia ventricular)',
        'Frecuencia cardíaca máxima alcanzada (bpm)',
        'Angina inducida por ejercicio (1=sí, 0=no)',
        'Depresión del segmento ST inducida por ejercicio',
        'Pendiente del segmento ST (1=ascendente, 2=plana, 3=descendente)',
        'Nº de vasos coloreados por fluoroscopia (0-3)',
        'Talasemia (3=normal, 6=defecto fijo, 7=defecto reversible)',
        'Diagnóstico (0=sano, 1-4=grados de enfermedad)'
    ],
    'Tipo esperado': [
        'int', 'int (cat)', 'int (cat)', 'float', 'float', 'int (cat)',
        'int (cat)', 'float', 'int (cat)', 'float', 'int (cat)', 'int',
        'int (cat)', 'int (target)'
    ]
})
display(data_dict)

In [ ]:
# ============================================
# TIPOS Y ESTADÍSTICOS
# ============================================
print('=== TIPOS ACTUALES ===')
print('(Todas float64 por NaN — se corregirá tras limpieza)')
print(df_raw.dtypes)
print(f'\n=== DIMENSIONES: {df_raw.shape} ===')
print(f'\n=== ESTADÍSTICOS DESCRIPTIVOS ===')
df_raw.describe().round(2)

In [ ]:
# ============================================
# ANÁLISIS DE VALORES NULOS
# ============================================
print('=== VALORES NULOS POR VARIABLE ===')
null_counts = df_raw.isnull().sum()
null_pct = (df_raw.isnull().sum() / len(df_raw) * 100).round(2)
null_analysis = pd.DataFrame({'Nulos': null_counts, '% Nulos': null_pct})
null_analysis = null_analysis[null_analysis['Nulos'] > 0].sort_values('% Nulos', ascending=False)
display(null_analysis)

print('\n=== VALORES NULOS POR HOSPITAL ===')
null_by_source = df_raw.groupby('source').apply(lambda x: x.isnull().sum()).T
null_by_source = null_by_source.loc[null_by_source.sum(axis=1) > 0]
display(null_by_source)

In [ ]:
# ============================================
# VISUALIZACIÓN DE NULOS
# ============================================
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

cmap_custom = sns.color_palette([LIGHT_BLUE, RED], as_cmap=True)
sns.heatmap(df_raw.drop(columns='source').isnull(), cbar=True, yticklabels=False,
            cmap=cmap_custom, ax=axes[0])
axes[0].set_title('Mapa de Valores Nulos')

null_pct_plot = (df_raw.drop(columns='source').isnull().sum() / len(df_raw) * 100)
null_pct_plot = null_pct_plot[null_pct_plot > 0].sort_values(ascending=True)
bars = axes[1].barh(null_pct_plot.index, null_pct_plot.values)
for bar, val in zip(bars, null_pct_plot.values):
    bar.set_color(RED if val > 30 else ORANGE if val > 5 else TEAL)
axes[1].set_title('% de Valores Nulos por Variable')
axes[1].set_xlabel('% Nulos')
for i, v in enumerate(null_pct_plot.values):
    axes[1].text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=10, color=NAVY)

plt.tight_layout()
plt.savefig(f'{FIGURES}01_missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

---
### 3. Análisis de importancia de variables con nulos elevados

Las variables `ca` (66%), `thal` (53%) y `slope` (34%) presentan un porcentaje elevado de nulos, concentrados en Hungarian, Switzerland y VA Long Beach. **Antes de decidir qué hacer, analizamos su importancia predictiva** usando Cleveland (prácticamente completo) con tres métodos complementarios.

In [ ]:
# ============================================
# IMPORTANCIA — 3 MÉTODOS (CLEVELAND)
# ============================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif

df_cleveland_check = df_raw[df_raw['source'] == 'Cleveland'].copy()
df_cleveland_check['target'] = (df_cleveland_check['num'] > 0).astype(int)

print('=== NULOS EN CLEVELAND (variables críticas) ===')
for col in ['ca', 'thal', 'slope']:
    print(f'  {col}: {df_cleveland_check[col].isnull().sum()} nulos de {len(df_cleveland_check)}')

df_clev = df_cleveland_check.dropna().copy()
X_clev = df_clev.drop(columns=['num', 'target', 'source'])
y_clev = df_clev['target']

corr_with_target = df_cleveland_check.drop(columns=['source']).corr()['target'].drop(['target', 'num'])
mi_scores = mutual_info_classif(X_clev, y_clev, random_state=42)
mi_series = pd.Series(mi_scores, index=X_clev.columns)
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_clev, y_clev)
rf_importance = pd.Series(rf.feature_importances_, index=X_clev.columns)

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
critical_vars = ['ca', 'thal', 'slope']

corr_sorted = corr_with_target.abs().sort_values(ascending=True)
colors_1 = [RED if v in critical_vars else NAVY for v in corr_sorted.index]
corr_sorted.plot(kind='barh', ax=axes[0], color=colors_1)
axes[0].set_title('Correlación de Pearson\n(relaciones lineales)')

mi_sorted = mi_series.sort_values(ascending=True)
colors_2 = [RED if v in critical_vars else NAVY for v in mi_sorted.index]
mi_sorted.plot(kind='barh', ax=axes[1], color=colors_2)
axes[1].set_title('Información Mutua\n(relaciones no lineales)')

rf_sorted = rf_importance.sort_values(ascending=True)
colors_3 = [RED if v in critical_vars else NAVY for v in rf_sorted.index]
rf_sorted.plot(kind='barh', ax=axes[2], color=colors_3)
axes[2].set_title('Importancia Random Forest\n(captura interacciones)')

plt.suptitle('Importancia de variables (Cleveland) — Rojo = alto % nulos en otros hospitales',
             fontsize=13, fontweight='bold', y=1.02, color=NAVY)
plt.tight_layout()
plt.savefig(f'{FIGURES}02_variable_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== RANKING COMBINADO ===')
ranking = pd.DataFrame({
    'Correlación': corr_with_target.abs().rank(ascending=False),
    'Info Mutua': mi_series.rank(ascending=False),
    'Random Forest': rf_importance.rank(ascending=False)
})
ranking['Media'] = ranking.mean(axis=1)
print(ranking.sort_values('Media').round(1))

---
### 4. Decisión fundamentada y limpieza

Los tres métodos confirman que **`thal`, `ca` y `slope` son las variables más predictivas**. Eliminarlas perdería el poder discriminativo del modelo.

Sin embargo, están vacías en **>95% de las observaciones** de Hungarian, Switzerland y VA. Imputar el 95% equivaldría a **fabricar datos, no a recuperarlos**.

**Estrategia:**
1. Conservar todas las variables
2. Exigir que `thal` (la #1 en importancia) tenga valor real
3. Imputar `ca` y `slope` con **moda por grupo de target** (más adecuado que KNN para categóricas)
4. Imputar variables continuas con **KNN (k=5)** excluyendo el target para evitar data leakage

In [ ]:
# ============================================
# PASO 1: BINARIZAR TARGET
# ============================================
df_raw['target'] = (df_raw['num'] > 0).astype(int)

print('=== TARGET ORIGINAL vs BINARIZADO ===')
print(f'\nOriginal (num):\n{df_raw["num"].value_counts().sort_index()}')
print(f'\nBinarizado (target):\n{df_raw["target"].value_counts()}')
print(f'Proporción con enfermedad: {df_raw["target"].mean():.2%}')

df_raw = df_raw.drop(columns=['num'])

fig, ax = plt.subplots(figsize=(8, 5))
counts = df_raw['target'].value_counts().sort_index()
ax.bar(['Sano (0)', 'Enfermo (1)'], counts.values, color=[TEAL, RED], edgecolor='white')
for i, v in enumerate(counts.values):
    ax.text(i, v + 5, f'{v} ({v/len(df_raw)*100:.1f}%)', ha='center', fontweight='bold', color=NAVY)
ax.set_title('Target Binarizado', color=NAVY)
ax.set_ylabel('Frecuencia')
plt.tight_layout()
plt.savefig(f'{FIGURES}03_target.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================
# PASO 2: FILTRADO — EXIGIR thal NO NULO
# ============================================
print('=== OBSERVACIONES CON thal COMPLETO ===')
for source in df_raw['source'].unique():
    df_s = df_raw[df_raw['source'] == source]
    complete = df_s['thal'].notna().sum()
    total = len(df_s)
    print(f'  {source}: {complete}/{total} con thal ({complete/total*100:.1f}%)')

rows_before = len(df_raw)
df_raw = df_raw[df_raw['thal'].notna()].copy()

print(f'\nRegistros antes: {rows_before}')
print(f'Registros después: {len(df_raw)} (eliminados: {rows_before - len(df_raw)})')
print(f'\nComposición:')
print(df_raw['source'].value_counts())
print(f'\nNulos restantes:')
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])

In [ ]:
# ============================================
# PASO 3: IMPUTACIÓN SEPARADA — CATEGÓRICAS vs CONTINUAS
# ============================================
# CORRECCIÓN: No usar KNN para categóricas (distancia euclidiana no aplica)
# CORRECCIÓN: No incluir target en la imputación (data leakage)
from sklearn.impute import KNNImputer

df_before_imputation = df_raw.copy()

# --- CATEGÓRICAS: imputar con moda por grupo de target ---
cat_to_impute = ['slope', 'ca', 'fbs', 'restecg', 'exang']
for col in cat_to_impute:
    if df_raw[col].isnull().any():
        for t in [0, 1]:
            mask = (df_raw['target'] == t) & (df_raw[col].isnull())
            if mask.any():
                mode_val = df_raw[(df_raw['target'] == t) & df_raw[col].notna()][col].mode()
                if len(mode_val) > 0:
                    df_raw.loc[mask, col] = mode_val.iloc[0]

# --- CONTINUAS: KNN sin incluir target ---
continuous_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
cols_with_null_cont = [c for c in continuous_cols if df_raw[c].isnull().any()]

if len(cols_with_null_cont) > 0:
    # Solo usar features continuas como referencia, NUNCA target
    imputer = KNNImputer(n_neighbors=5)
    df_raw[continuous_cols] = imputer.fit_transform(df_raw[continuous_cols])

print(f'Nulos antes: {df_before_imputation.drop(columns=["source"]).isnull().sum().sum()}')
print(f'Nulos después: {df_raw.drop(columns=["source"]).isnull().sum().sum()}')

# Visualización antes vs después
cols_imputed = [c for c in df_before_imputation.columns 
                if c not in ['source', 'target'] and df_before_imputation[c].isnull().any()]
if len(cols_imputed) > 0:
    n_plots = min(len(cols_imputed), 6)
    fig, axes = plt.subplots(1, n_plots, figsize=(5*n_plots, 5))
    if n_plots == 1:
        axes = [axes]
    for i, col in enumerate(cols_imputed[:n_plots]):
        axes[i].hist(df_before_imputation[col].dropna(), bins=25, alpha=0.5,
                     color=NAVY, label='Antes', density=True)
        axes[i].hist(df_raw[col], bins=25, alpha=0.5,
                     color=ORANGE, label='Después', density=True)
        axes[i].set_title(f'{col} ({df_before_imputation[col].isnull().sum()} imputados)')
        axes[i].legend(fontsize=8)
    plt.suptitle('Imputación — Antes vs Después', fontweight='bold', color=NAVY)
    plt.tight_layout()
    plt.savefig(f'{FIGURES}04_imputation.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ============================================
# PASO 4: CORRECCIÓN DE TIPOS Y RANGOS VÁLIDOS
# ============================================
# Redondear y convertir a int
int_cols = ['age', 'sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal', 'target']
for col in int_cols:
    df_raw[col] = df_raw[col].round().astype(int)

# Asegurar rangos válidos (KNN/moda pueden producir valores fuera de rango)
df_raw['cp'] = df_raw['cp'].clip(1, 4)
df_raw['restecg'] = df_raw['restecg'].clip(0, 2)
df_raw['slope'] = df_raw['slope'].clip(1, 3)
df_raw['ca'] = df_raw['ca'].clip(0, 3)
df_raw['sex'] = df_raw['sex'].clip(0, 1)
df_raw['fbs'] = df_raw['fbs'].clip(0, 1)
df_raw['exang'] = df_raw['exang'].clip(0, 1)

print('=== TIPOS CORREGIDOS ===')
print(df_raw.dtypes)
print(f'\nNulos: {df_raw.isnull().sum().sum()}')

---
### 5. Detección y tratamiento de outliers

In [ ]:
# ============================================
# DETECCIÓN DE OUTLIERS
# ============================================
fig, axes = plt.subplots(1, len(continuous_cols), figsize=(20, 5))
outlier_info = []
for i, col in enumerate(continuous_cols):
    axes[i].boxplot(df_raw[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor=LIGHT_BLUE, edgecolor=NAVY),
                    medianprops=dict(color=RED, linewidth=2),
                    flierprops=dict(marker='o', markerfacecolor=ORANGE, markersize=4),
                    whiskerprops=dict(color=NAVY), capprops=dict(color=NAVY))
    axes[i].set_title(col)
    Q1, Q3 = df_raw[col].quantile(0.25), df_raw[col].quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((df_raw[col] < Q1 - 1.5*IQR) | (df_raw[col] > Q3 + 1.5*IQR)).sum()
    outlier_info.append({'Variable': col, 'Outliers': n_out,
                         'Lím.inf': round(Q1-1.5*IQR,1), 'Lím.sup': round(Q3+1.5*IQR,1)})

plt.suptitle('Detección de Outliers', fontweight='bold', color=NAVY)
plt.tight_layout()
plt.savefig(f'{FIGURES}05_outliers.png', dpi=150, bbox_inches='tight')
plt.show()
display(pd.DataFrame(outlier_info))

In [ ]:
# ============================================
# WINSORIZACIÓN
# ============================================
df_before_outliers = df_raw.copy()
total_outliers_before = sum(d['Outliers'] for d in outlier_info)

for col in continuous_cols:
    Q1, Q3 = df_raw[col].quantile(0.25), df_raw[col].quantile(0.75)
    IQR = Q3 - Q1
    df_raw[col] = df_raw[col].clip(lower=Q1 - 1.5*IQR, upper=Q3 + 1.5*IQR)

# Nota: tras winsorizar, recalcular IQR puede revelar nuevos outliers marginales.
# Esto es esperado y no requiere iteraciones adicionales.
print(f'Outliers tratados con winsorización: {total_outliers_before}')

---
### 6. Exportación del dataset limpio

Se exporta **sin estandarizar**. La estandarización se aplicará en la fase de modelado (fit sobre train, transform sobre test) para evitar data leakage.

In [ ]:
# ============================================
# GUARDAR CSV — SIN ESTANDARIZAR
# ============================================
# source es trazabilidad, NO feature predictiva.
# Debe excluirse del modelado.
df_raw.to_csv('../data/processed/heart_disease_clean.csv', index=False)
print(f'Dataset guardado: {df_raw.shape}')
print(f'Nulos: {df_raw.isnull().sum().sum()}')

---
### 7. Transformaciones y análisis (sobre copia estandarizada)

In [ ]:
# ============================================
# ESTANDARIZACIÓN — SOLO PARA VISUALIZACIÓN Y ANÁLISIS
# No se aplica al CSV exportado.
# ============================================
from sklearn.preprocessing import StandardScaler

df_std = df_raw.copy()
scaler = StandardScaler()
df_std[continuous_cols] = scaler.fit_transform(df_std[continuous_cols])

print('=== ESTADÍSTICOS TRAS ESTANDARIZACIÓN (solo visualización) ===')
df_std[continuous_cols].describe().round(3)

In [ ]:
# ============================================
# VIF
# ============================================
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_vif = df_std[continuous_cols].copy()
vif_data = pd.DataFrame({
    'Variable': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
}).sort_values('VIF', ascending=False)

print('=== VIF ===')
display(vif_data)

fig, ax = plt.subplots(figsize=(10, 5))
colors_vif = [RED if v > 10 else ORANGE if v > 5 else TEAL for v in vif_data['VIF']]
ax.barh(vif_data['Variable'], vif_data['VIF'], color=colors_vif, edgecolor='white')
ax.axvline(x=5, color=ORANGE, linestyle='--', linewidth=1.5, label='Moderado (5)')
ax.axvline(x=10, color=RED, linestyle='--', linewidth=1.5, label='Alto (10)')
ax.set_title('VIF', color=NAVY)
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIGURES}06_vif.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================
# MATRIZ DE CORRELACIÓN
# ============================================
fig, ax = plt.subplots(figsize=(13, 10))
corr_matrix = df_std.drop(columns=['source']).corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title('Matriz de Correlación', color=NAVY)
plt.tight_layout()
plt.savefig(f'{FIGURES}07_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

---
### 8. Análisis Exploratorio Final

In [ ]:
# ============================================
# CONTINUAS POR TARGET
# ============================================
fig, axes = plt.subplots(1, len(continuous_cols), figsize=(22, 5))
for i, col in enumerate(continuous_cols):
    for t, color, label in [(0, TEAL, 'Sano'), (1, RED, 'Enfermo')]:
        axes[i].hist(df_raw[df_raw['target']==t][col], bins=25, alpha=0.5,
                     color=color, label=label, density=True)
    axes[i].set_title(col)
    axes[i].legend(fontsize=8)
plt.suptitle('Variables Continuas por Diagnóstico', fontweight='bold', color=NAVY)
plt.tight_layout()
plt.savefig(f'{FIGURES}08_continuous_by_target.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================
# CATEGÓRICAS POR TARGET
# ============================================
cat_analysis = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
axes = axes.flatten()
for i, col in enumerate(cat_analysis):
    ct = pd.crosstab(df_raw[col], df_raw['target'], normalize='index') * 100
    ct.plot(kind='bar', stacked=True, ax=axes[i], color=[TEAL, RED], edgecolor='white')
    axes[i].set_title(col)
    axes[i].set_ylabel('% pacientes')
    axes[i].legend(['Sano', 'Enfermo'], fontsize=8)
    axes[i].tick_params(axis='x', rotation=0)
plt.suptitle('Variables Categóricas vs Diagnóstico', fontweight='bold', color=NAVY)
plt.tight_layout()
plt.savefig(f'{FIGURES}09_categorical_by_target.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================
# RESUMEN FINAL
# ============================================
print('=' * 60)
print('       RESUMEN DE INGENIERÍA DEL DATO')
print('=' * 60)
print(f'Registros originales:  920 (4 hospitales)')
print(f'Registros finales:     {len(df_raw)}')
print(f'Variables:             {df_raw.shape[1]-2} features + target + source')
print(f'Nulos:                 {df_raw.isnull().sum().sum()}')
print(f'')
print(f'Procesos aplicados:')
print(f'  1. Combinación de 4 repositorios (UCI)')
print(f'  2. Análisis importancia (Pearson, MI, RF)')
print(f'  3. Filtrado: exigir thal no nulo')
print(f'  4. Imputación: moda para categóricas, KNN para continuas')
print(f'     (target excluido de imputación — sin data leakage)')
print(f'  5. Corrección de tipos y rangos válidos')
print(f'  6. Winsorización (IQR 1.5x)')
print(f'  7. VIF + correlación (sobre copia estandarizada)')
print(f'')
print(f'CSV exportado SIN estandarizar (se hará en modelado)')
print(f'source = trazabilidad, NO feature predictiva')
print('=' * 60)